In [6]:
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd


# -------------------------
# 1. Check probability validity
# -------------------------
def s_nan(y_prob, eps=1e-3):
    if not np.all(np.isfinite(y_prob)):
        return 0

    row_sums = np.sum(y_prob, axis=1)
    if np.all(np.abs(row_sums - 1.0) <= eps):
        return 1
    return 0


# -------------------------
# 2. Majority + JSD score
# -------------------------
def s_maj_jsd(y_true, y_pred, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_true))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)
    true_dist = np.bincount(y_true, minlength=n_classes) / len(y_true)

    # Majority score
    f_max = np.max(pred_dist)
    s_maj = 1 - (f_max - 1/n_classes) / (1 - 1/n_classes)

    # Jensen-Shannon divergence
    # jensenshannon() trả về căn bậc hai của JSD dùng log e
    js_distance = jensenshannon(true_dist, pred_dist)
    
    # Bình phương để lấy JSD (Divergence)
    js_divergence = js_distance**2
    s_jsd = 1 - (js_divergence / np.log(2))

    return s_maj, s_jsd


# -------------------------
# 3. Entropy score
# -------------------------
def s_ent(y_prob):
    N, K = y_prob.shape

    h = -np.sum(y_prob * np.log(y_prob + 1e-10), axis=1) / np.log(K)
    m = np.median(h)

    return 1 - min(abs(m - 0.5) / 0.5, 1.0)


# -------------------------
# 4. Drift score
# -------------------------
def s_drift(y_pred, ref_dist=None, n_classes=None):
    if n_classes is None:
        n_classes = len(np.unique(y_pred))

    pred_dist = np.bincount(y_pred, minlength=n_classes) / len(y_pred)

    if ref_dist is None:
        ref_dist = np.ones(n_classes) / n_classes

    # jensenshannon trả về căn bậc hai của JSD (Distance)
    js_distance = jensenshannon(pred_dist, ref_dist)
    
    # Bình phương để có JSD (Divergence)
    js_divergence = js_distance**2
    
    return 1 - (js_divergence / np.log(2))


# -------------------------
# 5. Efficiency score
# -------------------------
def s_eff(mask):
    return np.mean(mask)


# -------------------------
# 6. Leakage score (generic probe)
# -------------------------
def s_leak_1(mask, y_true, probe_model, test_size=0.3, random_state=40):

    X_train, X_test, y_train, y_test = train_test_split(
        mask, y_true,
        test_size=test_size,
        random_state=random_state,
        stratify=y_true
    )

    probe_model.fit(X_train, y_train)

    if hasattr(probe_model, "predict_proba"):
        y_score = probe_model.predict_proba(X_test)
    else:
        y_score = probe_model.decision_function(X_test)

    auc = roc_auc_score(y_test, y_score, multi_class="ovr")
    return 1 - auc


# -------------------------
# 7. Logistic leakage
# -------------------------
def s_leak(X, y, test_size=0.3, random_state=40):

    Xtr, Xte, ytr, yte = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    probe = LogisticRegression(max_iter=1000)
    probe.fit(Xtr, ytr)

    y_score = probe.predict_proba(Xte)

    auc = roc_auc_score(yte, y_score, multi_class="ovr")
    return float(1 - auc)

# -------------------------
# 8. Final geometric mean score
# -------------------------
def S_san_plus(values):
    values = np.clip(values, 1e-10, 1)
    return np.prod(values) ** (1 / len(values))


In [7]:
def compute_ref_dist(y_train):
    y = y_train.astype(int).to_numpy()
    n_classes = len(np.unique(y))
    return np.bincount(y, minlength=n_classes) / len(y)

def run_sanity_evaluation(
    *,
    path_1,
    path_2,
    version_name,
    train_name,
    n_classes=3
):
    print(
        f"[START] Sanity evaluation | "
        f"train={train_name} | version={version_name}"
    )

    rows = []

    # -----------------
    # Load train
    # -----------------
    train_df = pd.read_csv(f"{path_1}/{train_name}.csv")
    y_train = train_df["label_3"]
    ref_dist = compute_ref_dist(y_train)

    # -----------------
    # Loop phases
    # -----------------
    for phase in [1, 2, 3, 4]:
        print(f"  -> Running phase {phase}...")

        # ---- probability matrix
        prob_df = pd.read_csv(
            f"{path_2}/probability_matrix_Hybrid-Mask-{version_name}_phase{phase}.csv"
        )

        y_true = prob_df["y_true"].astype(int).to_numpy()
        y_pred = prob_df["y_pred"].astype(int).to_numpy()
        y_prob = prob_df[
            ["Prob_Class_0", "Prob_Class_1", "Prob_Class_2"]
        ].to_numpy()

        # ---- test data (for leakage & efficiency)
        test_df = pd.read_csv(f"{path_1}/test_{phase}.csv")
        X_test = test_df.drop(columns=["label_3"], errors="ignore")
        mask = X_test.notna().astype(int).values

        # ---- metrics
        smaj, sjsd = s_maj_jsd(y_true, y_pred, n_classes=n_classes)

        snan   = s_nan(y_prob)
        sent   = s_ent(y_prob)
        sdrift = s_drift(y_pred, ref_dist, n_classes=n_classes)
        sleak  = s_leak(mask, y_true)
        s_eff_ = s_eff(mask)

        s_san_plus = S_san_plus([
            snan,
            sjsd,
            sent,
            sdrift,
            s_eff_,
            sleak
        ])

        rows.append({
            "train_name": train_name,
            "version_name": version_name,
            "phase": phase,
            "snan": snan,
            "smaj": smaj,
            "sjsd": sjsd,
            "sent": sent,
            "sdrift": sdrift,
            "sleak": sleak,
            "s_san+": s_san_plus,
            "s_eff": s_eff_
        })

    print(
        f"[END] Sanity evaluation DONE | "
        f"rows={len(rows)} | "
        f"phases=4"
    )

    columns_order = [
        "train_name",
        "version_name",
        "phase",
        "snan",
        "smaj",
        "sjsd",
        "sent",
        "sdrift",
        "sleak",
        "s_san+",
        "s_eff"
    ]

    return pd.DataFrame(rows)[columns_order]


# Chạy với các version data

In [8]:
path_1 = "/kaggle/input/lo-dataset/Raw/Raw"

In [9]:
path_2 = "/kaggle/input/datasets/thuhng9/v0-mask/results"

## RNN

In [10]:
df_rnn = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="RNN",
    train_name="train_raw"
)
df_rnn

[START] Sanity evaluation | train=train_raw | version=RNN
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_raw,RNN,1,1,0.685435,0.732444,0.649238,0.732442,0.215232,0.625008,0.795163
1,train_raw,RNN,2,1,0.728741,0.710491,0.645371,0.710489,0.110112,0.525099,0.584368
2,train_raw,RNN,3,1,0.638858,0.638085,0.610309,0.638083,0.040952,0.394594,0.370955
3,train_raw,RNN,4,1,0.657068,0.646802,0.635604,0.646800,0.017710,0.300404,0.156058


In [11]:
df_rnn.to_csv("results_rnn.csv", index=False)

## Bilstm

In [12]:
df_bilstm = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="BiLSTM",
    train_name="train_raw"
)
df_bilstm

[START] Sanity evaluation | train=train_raw | version=BiLSTM
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_raw,BiLSTM,1,1,0.000000,0.998217,4.523169e-09,0.998217,0.215232,0.030284,0.795163
1,train_raw,BiLSTM,2,1,0.000000,0.998217,1.345551e-08,0.998217,0.110112,0.030854,0.584368
2,train_raw,BiLSTM,3,1,0.000032,0.998296,7.124104e-07,0.998296,0.040952,0.047005,0.370955
3,train_raw,BiLSTM,4,1,0.001813,0.999331,7.975770e-07,0.999332,0.017710,0.036067,0.156058


In [13]:
df_bilstm.to_csv("results_bilstm.csv", index=False)

## LSTM

In [14]:
df_lstm = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="LSTM",
    train_name="train_raw"
)
df_lstm

[START] Sanity evaluation | train=train_raw | version=LSTM
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_raw,LSTM,1,1,0.000000,0.998217,0.000009,0.998217,0.215232,0.108016,0.795163
1,train_raw,LSTM,2,1,0.000000,0.998217,0.000029,0.998217,0.110112,0.111075,0.584368
2,train_raw,LSTM,3,1,0.000103,0.998413,0.000841,0.998414,0.040952,0.152816,0.370955
3,train_raw,LSTM,4,1,0.002252,0.999487,0.000978,0.999487,0.017710,0.118008,0.156058


In [15]:
df_lstm.to_csv("results_lstm.csv", index=False)

## V4 (Median RadiusSMOTE)

In [16]:
df_gru = run_sanity_evaluation(
    path_1=path_1,
    path_2=path_2,
    version_name="GRU",
    train_name="train_raw"
)
df_gru

[START] Sanity evaluation | train=train_raw | version=GRU
  -> Running phase 1...
  -> Running phase 2...
  -> Running phase 3...
  -> Running phase 4...
[END] Sanity evaluation DONE | rows=4 | phases=4


,train_name,version_name,phase,snan,smaj,sjsd,sent,sdrift,sleak,s_san+,s_eff
0,train_raw,GRU,1,1,0.000000,0.998217,0.000024,0.998217,0.215232,0.126212,0.795163
1,train_raw,GRU,2,1,0.000000,0.998217,0.000062,0.998217,0.110112,0.125792,0.584368
2,train_raw,GRU,3,1,0.000077,0.998375,0.000537,0.998375,0.040952,0.141812,0.370955
3,train_raw,GRU,4,1,0.005175,0.999553,0.000642,0.999553,0.017710,0.110021,0.156058


In [17]:
df_gru.to_csv("results_gru.csv", index=False)